In [5]:
# Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted!


In [6]:
!pip install -q unsloth

In [7]:
import unsloth
from unsloth import FastLanguageModel

print("Unsloth imported successfully!")

Unsloth imported successfully!


In [8]:
import torch
import transformers
import datasets
import peft
import accelerate
import unsloth

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("unsloth:", unsloth.__version__)

torch: 2.11.0+cu128
transformers: 5.5.0
datasets: 4.3.0
peft: 0.20.0
accelerate: 1.14.0
unsloth: 2026.9.11


In [9]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.9Gi       1.1Gi        14Mi       6.9Gi       7.7Gi
Swap:             0B          0B          0B


In [10]:
!nvidia-smi

Sun Sep 27 13:25:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             28W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
# Check hardware

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No NVIDIA GPU detected.")

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


# Dataset Dict

In [12]:
# Load Existing Prepared Data

from datasets import load_dataset

data_path = "/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data"

prepared_dataset = load_dataset(
    "json",
    data_files={
        "train": f"{data_path}/train.jsonl",
        "test": f"{data_path}/test.jsonl"
    }
)

print(prepared_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 23813
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5954
    })
})


In [13]:
# Create Validation 10% and Final Test Split 10%

test_split = prepared_dataset["test"].train_test_split(
    test_size=0.5,
    seed=42
)

print(test_split)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2977
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2977
    })
})


In [14]:
# Train 80%, Validation 10% and Test 10%

from datasets import DatasetDict

final_dataset = DatasetDict({
    "train": prepared_dataset["train"],
    "validation": test_split["train"],
    "test": test_split["test"]
})

print(final_dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 23813
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2977
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2977
    })
})


# Create Qwen messages

In [15]:
SYSTEM_PROMPT = (
    "You are a cybersecurity expert specializing in phishing detection. "
    "Analyze the provided email, message, or URL. "
    "Determine whether it is a phishing attempt or a legitimate communication. "
    "Respond with only one word: 'phishing' or 'legitimate'."
)

def convert_to_qwen_messages(sample):

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": sample["input"]
            },
            {
                "role": "assistant",
                "content": sample["output"]
            }
        ]
    }

In [16]:
qwen_dataset = final_dataset.map(
    convert_to_qwen_messages
)

Map:   0%|          | 0/23813 [00:00<?, ? examples/s]

Map:   0%|          | 0/2977 [00:00<?, ? examples/s]

Map:   0%|          | 0/2977 [00:00<?, ? examples/s]

In [17]:
print(qwen_dataset["train"][0]["messages"])

[{'content': "You are a cybersecurity expert specializing in phishing detection. Analyze the provided email, message, or URL. Determine whether it is a phishing attempt or a legitimate communication. Respond with only one word: 'phishing' or 'legitimate'.", 'role': 'system'}, {'content': "Subject: barrister thambo\n\nBody: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s\r\nhowever the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?\r\nexpecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com\r\nregard ' s\r\nbarrister andrew thambo", 'role': 'user'}, {'conten

In [18]:
example = qwen_dataset["train"][0]["messages"]

print("========== SYSTEM ==========")
print(example[0]["content"])

print("\n========== USER ==========")
print(example[1]["content"])

print("\n========== ASSISTANT ==========")
print(example[2]["content"])

========== SYSTEM ==========
You are a cybersecurity expert specializing in phishing detection. Analyze the provided email, message, or URL. Determine whether it is a phishing attempt or a legitimate communication. Respond with only one word: 'phishing' or 'legitimate'.

========== USER ==========
Subject: barrister thambo

Body: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s
however the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?
expecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com
regard ' s
barrister andrew thambo

========== ASSISTANT ==========

# Load Qwen + tokenizer with Unsloth

In [19]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    load_in_4bit=True
)

print("Model loaded successfully!")

==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!


In [20]:
#Test Qwen Chat Template

example_messages = qwen_dataset["train"][0]["messages"]

formatted_text = tokenizer.apply_chat_template(
    example_messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

<|im_start|>system
You are a cybersecurity expert specializing in phishing detection. Analyze the provided email, message, or URL. Determine whether it is a phishing attempt or a legitimate communication. Respond with only one word: 'phishing' or 'legitimate'.<|im_end|>
<|im_start|>user
Subject: barrister thambo

Body: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s
however the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?
expecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com
regard ' s
barrister andrew thambo<|im_end|>
<|im_start|>assistant
phishing<|i

In [21]:
# Baseline Inference (Before Fine-Tuning)

# Take one example from the test set
sample = qwen_dataset["test"][0]

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": sample["messages"][1]["content"]
    }
]

# Format the prompt using Qwen's chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate prediction
output = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Extract only the generated answer
generated_ids = output[0][inputs.input_ids.shape[1]:]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

print("EMAIL:")
print(sample["messages"][1]["content"])

print("\nEXPECTED LABEL:")
print(sample["messages"][2]["content"])

print("\nSTARTING MODEL PREDICTION:")
print(prediction)

Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EMAIL:
Subject: abb contract

Body: dave ,
i am still diving into the depths of tex - mex , but cannot see a reason not to proceed with signing the abb amendment as described in salsa ( 4 ) , as this amendment does not cost us anything up front and extends our build option to the end of the year .
there are a number of risks associated with the presidential permit process , the possible imposition of an open season by ferc , relationship with cfe , bpub infrastructure enhancements and service charge structure , etc . which we should discuss prior to waiving our conditions precedent under the deacero contract on july 31 .
note that if we sign the abb amendment and waive the conditions precedent along with deacero on july 31 , we will want to immediately issue a pre - notice of the notice to proceed in order to ensure completion within the 502 day deacero commitment .
dave

EXPECTED LABEL:
legitimate

STARTING MODEL PREDICTION:
phishing


# LoRA with Unsloth

In [22]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    bias="none",
    use_gradient_checkpointing="unsloth"
)

print("LoRA configured successfully!")

model.print_trainable_parameters()

Unsloth 2026.9.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA configured successfully!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


# Tokenization

In [23]:
# Tokenization and Training Preparation

def tokenize_messages(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return tokenizer(
        text,
        truncation=True,
        max_length=1024
    )


tokenized_dataset = qwen_dataset.map(
    tokenize_messages,
    batched=False,
    remove_columns=[
        "messages",
        "instruction",
        "input",
        "output"
    ]
)

print(tokenized_dataset)

Map:   0%|          | 0/23813 [00:00<?, ? examples/s]

Map:   0%|          | 0/2977 [00:00<?, ? examples/s]

Map:   0%|          | 0/2977 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 23813
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2977
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2977
    })
})


# Training Setup

In [24]:
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["validation"]

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/PhishDetectAI/training_results_correct",

    num_train_epochs=0.35,
    learning_rate=2e-4,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,

    fp16=True,

    logging_steps=10,
    eval_strategy="epoch",

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    lr_scheduler_type="cosine",

    remove_unused_columns=False,

    report_to="none"
)

# Create Trainer

In [25]:
import trl
print("trl:", trl.__version__)

trl: 0.24.0


In [26]:
#Create SFT Trainer

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    dataset_text_field=None,

    max_seq_length=1024,

    data_collator=data_collator,

    args=training_args
)

print("Trainer created successfully!")

Trainer created successfully!


# Start Tranning

In [27]:
#Execute Training

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 23,813 | Num Epochs = 1 | Total steps = 4,168
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is

Epoch,Training Loss,Validation Loss
0,1.595799,1.709459


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/training_results_correct/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/training_results_correct/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/training_results_correct/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/training_results_correct/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/training_results_correct/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/training_results_correct/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/t

TrainOutput(global_step=4168, training_loss=1.8613719331554626, metrics={'train_runtime': 4825.7522, 'train_samples_per_second': 1.727, 'train_steps_per_second': 0.864, 'total_flos': 2.5451345524706304e+16, 'train_loss': 1.8613719331554626, 'epoch': 0.35006089110989796})

In [29]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/PhishDetectAI/training_results_correct"

print(os.listdir(OUTPUT_DIR))

['README.md', 'checkpoint-4000', 'checkpoint-4168']


In [32]:
# Save Final Model

final_model_path = "/content/drive/MyDrive/PhishDetectAI/models/qwen_phishing_unsloth_correct"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("Model saved to:")
print(final_model_path)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PhishDetectAI/models/qwen_phishing_unsloth_correct/tokenizer_config.json.


Model saved to:
/content/drive/MyDrive/PhishDetectAI/models/qwen_phishing_unsloth_correct


### Post Fine-Tuning Inference

Let's perform an inference with the fine-tuned model to see its performance.

### Model Evaluation: Accuracy, Recall, Precision, and F1-score

In [33]:
from transformers import pipeline

# Create a pipeline for text generation with the fine-tuned model
# We'll use the 'text-generation' task, as Qwen2.5-1.5B-Instruct is an instruction-tuned model.
finetuned_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8 else torch.float16, # Use bfloat16 for Ampere and newer, else float16
    device_map="auto"
)

# Take one example from the test set for inference
# We'll use a different sample from the one used in baseline inference to avoid bias
sample_test_idx = 1 # Using index 1 for a new sample
sample = qwen_dataset["test"][sample_test_idx]

# Prepare the messages for the fine-tuned model
messages_for_inference = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": sample["messages"][1]["content"]
    }
]

# Apply the chat template for generation
formatted_input = tokenizer.apply_chat_template(
    messages_for_inference,
    tokenize=False,
    add_generation_prompt=True
)

# Generate prediction using the pipeline
outputs = finetuned_pipeline(
    formatted_input,
    max_new_tokens=5, # We expect only 'phishing' or 'legitimate'
    do_sample=False,  # For reproducible results
    eos_token_id=tokenizer.eos_token_id
)

# Extract the generated text
# The pipeline returns a list of dictionaries; we need the generated_text from the first item
# Then, we strip the input prompt to get only the model's prediction
generated_text = outputs[0]["generated_text"]
prediction_finetuned = generated_text.replace(formatted_input, "").strip()

print("EMAIL:")
print(sample["messages"][1]["content"])

print("\nEXPECTED LABEL:")
print(sample["messages"][2]["content"])

print("\nFINE-TUNED MODEL PREDICTION:")
print(prediction_finetuned)

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'do_sample', 'cache_implementation', 'eos_token_id', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EMAIL:
Subject: = ? utf - 8 ? q ? are you pimped with ? =

Body: = ? utf - 8 ? q ? a rolex watch ? ? =
veritable replications wristwatches
we offer next trademarks available also :
bvlgari
chopard
iwc
zenith
omega


EXPECTED LABEL:
phishing

FINE-TUNED MODEL PREDICTION:
phishing


In [36]:
sample = qwen_dataset["test"][0]

messages_for_inference = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": sample["messages"][1]["content"]
    }
]

formatted_input = tokenizer.apply_chat_template(
    messages_for_inference,
    tokenize=False,
    add_generation_prompt=True
)

outputs = finetuned_pipeline(
    formatted_input,
    max_new_tokens=10,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

print("ACTUAL LABEL:")
print(sample["messages"][2]["content"])

print("\nGENERATED TEXT:")
print(outputs[0]["generated_text"])

print("\n--------------------")

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ACTUAL LABEL:
legitimate

GENERATED TEXT:
<|im_start|>system
You are a cybersecurity expert specializing in phishing detection. Analyze the provided email, message, or URL. Determine whether it is a phishing attempt or a legitimate communication. Respond with only one word: 'phishing' or 'legitimate'.<|im_end|>
<|im_start|>user
Subject: abb contract

Body: dave ,
i am still diving into the depths of tex - mex , but cannot see a reason not to proceed with signing the abb amendment as described in salsa ( 4 ) , as this amendment does not cost us anything up front and extends our build option to the end of the year .
there are a number of risks associated with the presidential permit process , the possible imposition of an open season by ferc , relationship with cfe , bpub infrastructure enhancements and service charge structure , etc . which we should discuss prior to waiving our conditions precedent under the deacero contract on july 31 .
note that if we sign the abb amendment and waive

In [37]:
import torch
from transformers import pipeline
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. Load the fine-tuned Qwen model
# ============================================================

final_model_path = "/content/drive/MyDrive/PhishDetectAI/models/qwen_phishing_unsloth_correct"

model, tokenizer = FastLanguageModel.from_pretrained(
    final_model_path,
    max_seq_length=1024,
    load_in_4bit=True,
)

# ============================================================
# 2. Create text-generation pipeline
# ============================================================

finetuned_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=(
        torch.bfloat16
        if torch.cuda.is_available()
        and torch.cuda.get_device_properties(0).major >= 8
        else torch.float16
    ),
    device_map="auto"
)

# ============================================================
# 3. Prepare lists for predictions and actual labels
# ============================================================

predictions = []
references = []

# ============================================================
# 4. Evaluate every email in the TEST dataset
# ============================================================

for sample_idx in tqdm(
    range(len(qwen_dataset["test"])),
    desc="Evaluating model"
):

    sample = qwen_dataset["test"][sample_idx]

    # --------------------------------------------------------
    # Prepare the conversation
    # --------------------------------------------------------

    messages_for_inference = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": sample["messages"][1]["content"]
        }
    ]

    # --------------------------------------------------------
    # Apply Qwen chat template
    # --------------------------------------------------------

    formatted_input = tokenizer.apply_chat_template(
        messages_for_inference,
        tokenize=False,
        add_generation_prompt=True
    )

    # --------------------------------------------------------
    # Generate prediction
    # --------------------------------------------------------

    outputs = finetuned_pipeline(
        formatted_input,
        max_new_tokens=5,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        return_full_text=False
    )

    # --------------------------------------------------------
    # Extract generated answer
    # --------------------------------------------------------

    generated_text = outputs[0]["generated_text"].strip().lower()

    # --------------------------------------------------------
    # Convert generated text into a valid label
    # --------------------------------------------------------

    if "phishing" in generated_text:
        prediction = "phishing"

    elif "legitimate" in generated_text:
        prediction = "legitimate"

    else:
        prediction = "invalid"

    # --------------------------------------------------------
    # Get actual label
    # --------------------------------------------------------

    actual_label = sample["messages"][2]["content"].strip().lower()

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    predictions.append(prediction)
    references.append(actual_label)


# ============================================================
# 5. Check predictions
# ============================================================

print("\n" + "=" * 60)
print("PREDICTION CHECK")
print("=" * 60)

print("\nUnique predictions:")
print(set(predictions))

print("\nUnique actual labels:")
print(set(references))

print("\nNumber of predictions:", len(predictions))
print("Number of references:", len(references))


# ============================================================
# 6. Count invalid predictions
# ============================================================

invalid_count = predictions.count("invalid")

print("\nInvalid predictions:", invalid_count)


# ============================================================
# 7. Calculate metrics
# ============================================================

# Only calculate classification metrics using valid predictions
valid_indices = [
    i for i, prediction in enumerate(predictions)
    if prediction in ["phishing", "legitimate"]
]

valid_predictions = [
    predictions[i]
    for i in valid_indices
]

valid_references = [
    references[i]
    for i in valid_indices
]

accuracy = accuracy_score(
    valid_references,
    valid_predictions
)

precision = precision_score(
    valid_references,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

recall = recall_score(
    valid_references,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

f1 = f1_score(
    valid_references,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)


# ============================================================
# 8. Print main results
# ============================================================

print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)

print(f"\nTest samples: {len(qwen_dataset['test'])}")
print(f"Valid predictions: {len(valid_predictions)}")
print(f"Invalid predictions: {invalid_count}")

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


# ============================================================
# 9. Detailed classification report
# ============================================================

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        valid_references,
        valid_predictions,
        labels=["legitimate", "phishing"],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 10. Confusion matrix
# ============================================================

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(
    valid_references,
    valid_predictions,
    labels=["legitimate", "phishing"]
)

print("\n                 Predicted")
print("              Legitimate  Phishing")
print(f"Actual Legitimate   {cm[0][0]:5d}      {cm[0][1]:5d}")
print(f"Actual Phishing     {cm[1][0]:5d}      {cm[1][1]:5d}")

==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating model:   0%|          | 0/2977 [00:00<?, ?it/s]

Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 1458]) with length 1458 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to


PREDICTION CHECK

Unique predictions:
{'phishing', 'legitimate', 'invalid'}

Unique actual labels:
{'phishing', 'legitimate'}

Number of predictions: 2977
Number of references: 2977

Invalid predictions: 241

FINAL EVALUATION RESULTS

Test samples: 2977
Valid predictions: 2736
Invalid predictions: 241

Accuracy:  0.9905
Precision: 0.9862
Recall:    0.9938
F1 Score:  0.9900

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  legitimate     0.9944    0.9875    0.9909      1437
    phishing     0.9862    0.9938    0.9900      1299

    accuracy                         0.9905      2736
   macro avg     0.9903    0.9907    0.9905      2736
weighted avg     0.9905    0.9905    0.9905      2736


CONFUSION MATRIX

                 Predicted
              Legitimate  Phishing
Actual Legitimate    1419         18
Actual Phishing         8       1291


In [38]:
print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1 Score:  {f1 * 100:.2f}%")

Accuracy:  99.05%
Precision: 98.62%
Recall:    99.38%
F1 Score:  99.00%
